# wsp Colab runner

This notebook is the GPU server of the run on the home server that uses `--gpu colab`. The worker started at the
bottom fetches each forward pass from the run's relay over HTTPS, through the server's Cloudflare tunnel. It runs the
pass on this GPU with the same `gpu_server.py` as the home server and sends the outputs back. Nothing connects to
this notebook.

**Once:** in the key icon (Secrets) on the left, add a secret named `WSP_KEY` and turn on its notebook access. Its
value is the relay's key: on the PC, run `ssh root@<server> cat /opt/wsp/secrets/colab/key`.

**Each time** the run's driver logs "waiting for the Colab worker":
1. **Runtime → Change runtime type → A100 GPU.**
2. **Runtime → Run all.**
3. Keep this tab open and the computer awake. The last cell keeps running.

When the run no longer needs the GPU (it ended, it waits for a Claude usage limit, or it went
`--gpu-idle-minutes` without a pass), the worker **disconnects and deletes this runtime**, so it stops using
compute units. The run's next forward pass then waits until you run this notebook again.
Set `STOP_WHEN_IDLE = False` to keep the runtime instead.

In [ ]:
URL = "https://wsp-colab.example.com"    # the tunnel's public hostname for the relay (README, Colab runner)
STOP_WHEN_IDLE = True                   # disconnect this runtime when the run no longer needs it (saves units)

## 1. The GPU and its driver

In [ ]:
import subprocess
q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                   capture_output=True, text=True)
assert q.returncode == 0, "no GPU: Runtime -> Change runtime type -> A100 GPU"
gpu_name, gpu_mem, driver = [x.strip() for x in q.stdout.splitlines()[0].split(",")]
print(f"{gpu_name}, {gpu_mem}, driver {driver}")
if "A100" not in gpu_name:
    print("Note: not an A100. The run still works, but with this GPU's speed and unit price.")
DRIVER_MAJOR = int(driver.split(".")[0])

## 2. The relay's key and address

Reads `WSP_KEY` from Colab's secrets, then waits until the relay answers. It starts at the run's next forward
pass, and after 30 minutes without one the runtime is stopped.

In [ ]:
import sys, time, requests
from google.colab import userdata
try:
    KEY = userdata.get("WSP_KEY")
except Exception as e:
    raise SystemExit(f"Add the secret WSP_KEY (key icon on the left) and turn on its notebook access ({e})")
HEADERS = {}
try:  # optional: a Cloudflare Access service token, if the hostname is behind Access
    HEADERS = {"CF-Access-Client-Id": userdata.get("WSP_CF_ID"), "CF-Access-Client-Secret": userdata.get("WSP_CF_SECRET")}
except Exception:
    pass
auth = dict(HEADERS, Authorization=f"Bearer {KEY}")
t0 = time.time()
while True:
    try:
        r = requests.get(f"{URL}/v1/worker", headers=auth, timeout=60)
        if r.status_code == 200:
            break
        if r.status_code == 401:
            raise SystemExit("The relay refused the key: check the WSP_KEY secret")
        why = f"HTTP {r.status_code}"
        if r.status_code == 403:
            why += " from Cloudflare: Access without the WSP_CF_ID/WSP_CF_SECRET secrets, or Bot Fight Mode"
        elif r.status_code in (502, 530):
            why += ": no run is using Colab right now"
    except requests.RequestException as e:
        why = type(e).__name__
    if time.time() - t0 > 1800:
        if STOP_WHEN_IDLE:
            from google.colab import runtime
            runtime.unassign()
        raise SystemExit("The relay did not answer for 30 minutes")
    print(f"{time.strftime('%H:%M:%S')} the relay is not up yet ({why}); trying again in 30 s", flush=True)
    time.sleep(30)
open("/content/colab_worker.py", "w").write(r.text)
print("the relay answered; worker code fetched")

## 3. vLLM 0.32.0, as on the home server

This takes 3 to 6 minutes. vLLM's default build needs a CUDA 13 driver (580 or newer). On an older driver the
cell installs the CUDA 12.9 build instead.

In [ ]:
if DRIVER_MAJOR >= 580:
    cmd = "vllm==0.32.0 transformers==5.19.0"
else:
    cmd = ("https://github.com/vllm-project/vllm/releases/download/v0.32.0/"
           "vllm-0.32.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl transformers==5.19.0 "
           "--extra-index-url https://download.pytorch.org/whl/cu129")
r = subprocess.run(f"{sys.executable} -m pip install -q {cmd}", shell=True, capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-4000:])
assert r.returncode == 0, "pip install failed (see above)"
check = subprocess.run([sys.executable, "-c", "import vllm, torch, transformers; "
                        "print('vllm', vllm.__version__, '| torch', torch.__version__, '| transformers', "
                        "transformers.__version__, '| CUDA available:', torch.cuda.is_available())"],
                       capture_output=True, text=True)
print(check.stdout, check.stderr[-2000:])
assert check.returncode == 0 and "CUDA available: True" in check.stdout, "vLLM or torch does not work here"

## 4. The worker

It runs until the run releases the GPU. Its log shows each pass. Stop it with the stop button. The run's driver then
sees the GPU server as gone, and the next forward pass waits for this notebook.

In [ ]:
import importlib
sys.path.insert(0, "/content")
import colab_worker
importlib.reload(colab_worker)
colab_worker.main(URL, KEY, stop_when_idle=STOP_WHEN_IDLE, headers=HEADERS)